# Verify ASR inference end-to-end (Colab + free T4 GPU)

Runs the production inference pipeline against 5 held-out test samples per
model and compares each transcription to the prediction stored in
`final_test_results.json`. **If this notebook passes, the Docker image will
almost certainly work** — the inference path is the same.

Before doing anything that costs money (Docker Hub build, RunPod endpoint),
run this once.

**Runtime → Change runtime type → T4 GPU (free)** before running.

Expected time: ~15 min for the first cell (NeMo install + checkpoint
download), then ~2 min for each variant's verification.

## 1. Verify GPU is available

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count: ", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:           ", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU. Change runtime type to T4 GPU and run again."
    )

## 2. Clone the repo (feature branch)

Pulls the `feat/live-asr-demo-recording` branch which has the verifier script,
the conv_asr patch, and the User-Agent fixes for R2 downloads.

In [ ]:
%%bash
set -e
if [ ! -d /content/amchi_asr ]; then
  git clone --branch feat/live-asr-demo-recording \
    https://github.com/milind-kopikar/amchi_asr.git \
    /content/amchi_asr
fi
ls /content/amchi_asr | head -20

In [ ]:
import os, sys
os.chdir("/content/amchi_asr")
sys.path.insert(0, "/content/amchi_asr")
print("cwd:", os.getcwd())

## 3. Install dependencies

Mirrors what the Docker image installs — NeMo (asr only, not all), librosa,
jiwer, runpod SDK. The `--quiet` keeps Colab output short.

In [ ]:
%%bash
set -e
pip install --quiet --upgrade pip

# The problem we're solving:
#   Colab ships PyTorch 2.12, which removed the `deprecated=` kwarg from
#   torch._dynamo.config.Config(...). Code inside torch itself passes that
#   kwarg, so importing any nemo.collections.asr module dies with:
#     TypeError: Config() got an unexpected keyword argument 'deprecated'
#
# Naive fix (pin torch==2.5.1, then `pip install nemo_toolkit[asr]`) doesn't
# work because nemo's resolver bumps torch back up. Pinning nemo to an old
# version (e.g. 2.0.0) avoids that, but then nemo pulls in an old pyarrow
# that has no Python 3.12 wheel and fails to build from source.
#
# Working strategy: let pip install the LATEST nemo with all its proper
# Python 3.12-compatible deps, then forcibly downgrade torch (and only
# torch) with --force-reinstall --no-deps. Nemo's pure-Python code paths
# work fine against torch 2.5.1 — the version requirement is a resolver
# preference, not a hard ABI dependency.

# Step 1: install nemo + utilities normally. pip will bump torch to 2.12
# here; that's fine, we'll undo it in step 2.
pip install --quiet \
  "nemo_toolkit[asr]" \
  librosa omegaconf jiwer google-genai runpod \
  --ignore-installed blinker

# Step 2: forcibly downgrade ONLY torch/torchaudio/torchvision to 2.5.1,
# leaving every other package nemo brought in untouched.
pip install --quiet --force-reinstall --no-deps \
  "torch==2.5.1" "torchaudio==2.5.1" "torchvision==0.20.1" \
  --index-url https://download.pytorch.org/whl/cu121

# Step 3: HARD sanity check — fail loudly if torch is still 2.12.
python3 - <<'PY'
import torch
v = torch.__version__
print("torch:", v)
assert v.startswith("2.5."), (
    f"torch did not downgrade — still on {v}. Pip may have skipped the "
    "force-reinstall. Try: pip uninstall -y torch torchaudio torchvision "
    "then rerun this cell."
)
PY

echo done
echo
echo "IMPORTANT: After this cell finishes, click"
echo "  Runtime -> Disconnect and delete runtime"
echo "then reconnect and Run all from cell 1. The torch 2.12 already imported"
echo "in this kernel will NOT be replaced live; only a fresh runtime picks up"
echo "the pinned torch 2.5.1."

## 4. Apply the conv_asr patch

The fine-tuned hybrid CTC/RNNT checkpoint needs a patched `conv_asr.py` to
load. The Dockerfile does this — we replicate it here.

In [ ]:
import shutil
import nemo.collections.asr.modules.conv_asr as m
src = "/content/amchi_asr/patches/conv_asr_fixed.py"
shutil.copy(src, m.__file__)
print("Patched", m.__file__)

## 5. Run the staged smoke test

Verifies the dependency installs all wired up before the slow steps. If this
fails, fix the install rather than continuing.

In [ ]:
!python scripts/runpod_smoke.py --check imports --variant deaf
!python scripts/runpod_smoke.py --check patch   --variant deaf
!python scripts/runpod_smoke.py --check handler --variant deaf

## 6. Build the Amchi Konkani dictionary

The Amchi handler needs a Konkani dictionary JSON baked at
`data/amchi_konkani_dict.json`. Build it from the Railway API.

In [ ]:
!python scripts/build_amchi_dict.py --out data/amchi_konkani_dict.json
!python scripts/runpod_smoke.py --check dictionary --variant amchi

## 7. Verify the **Deaf Speech** model (canary)

Downloads the DS-D checkpoint (~1.4 GB) from R2, downloads 5 held-out test
audio samples, runs inference, compares to stored predictions.

Expected: all 5 samples within CER ≤ 0.05 of the stored prediction.

In [ ]:
!python scripts/verify_inference.py --variant deaf --num-samples 5

## 8. Verify the **Amchi Konkani** model

Only proceed if step 7 passed. Downloads the Run S checkpoint (~480 MB), then
5 Konkani test audio samples.

In [ ]:
!python scripts/verify_inference.py --variant amchi --num-samples 5

## What to do if a sample fails

If any sample shows `[FAIL]` in the result table, that's a real pipeline
regression. Look at the printed `stored:` vs `fresh:` lines:

- **stored is non-empty but fresh is empty** → checkpoint loaded with wrong
  decoding strategy; check `change_decoding_strategy(decoder_type='ctc')`.
- **fresh is garbage / random Devanagari** → conv_asr patch didn't apply; rerun cell 4.
- **fresh has mostly-right Devanagari but off by 1-2 characters** → likely
  benign CUDA float-ordering noise — increase `--tolerance` to 0.08 and re-run.
- **fresh is mostly correct but post-processed differently** → Gemini
  post-processing differences; not a model regression.

## Next step

If both cells passed, the inference pipeline is verified. You can proceed to
the GitHub Actions Docker build with high confidence the image will work.

See `docs/RUNPOD_GITHUB_ACTIONS_SETUP.md`.